# [LAB] LLM API 실습

같은 질문 "LLM이 무엇인지 3문장으로 설명해줘."를 세 가지 방법으로 LLM에 보내 봅니다. 방법이 바뀔 때마다 무엇이 편해지는지 비교하는 것이 이 노트북의 목표입니다.

- **2장 요청을 직접 보내기**: 주소·헤더·요청 본문을 손으로 만들어 보냅니다. 질문의 답 길이를 바꿔 응답의 토큰 수(`usage`)를 비교합니다.
- **3장 SDK로 묻기**: 도구(SDK)가 요청을 대신 보냅니다. 한 번 묻기, 이어서 묻기, 조각으로 받기를 실행합니다. 질문을 바꿔, 앞 대화가 있을 때와 없을 때 답이 어떻게 다른지 봅니다.
- **4장 LangChain으로 호출하기**: 같은 일을 `get_llm()` 한 줄로 준비한 모델로 합니다. 시스템 메시지를 바꿔 답의 모양을 보고, 조각 사이에 표시를 넣어 조각의 크기를 봅니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. 실습 방법
- 2. 요청을 직접 보내기
  - 2.1 요청 본문 만들기
  - 2.2 보내고 응답 받기
  - 2.3 참고: 다른 요청 방식(Responses API)
- 3. SDK로 한 번 묻기, 이어 묻기, 조각으로 받기
  - 3.1 한 번 묻기
  - 3.2 이어서 묻기
  - 3.3 조각으로 받기
  - 3.4 참고: temperature와 top_p
- 4. LangChain으로 호출하기
  - 4.1 역할 메시지로 한 번에 받기
  - 4.2 조각으로 받기
- 5. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/01-llm-api/01_requests.py`·`02_openai.py`·`03_langchain.py`. 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다.
- 원본에는 환경에 따라 주소·모델·인증을 고르는 분기가 있습니다(`LLM_PROVIDER`). openai 분기는 사외 준비용으로 우리가 더한 것이라 수강생 글에서는 다루지 않습니다. 2장의 보이는 셀(요청 본문)에 `provider`·`gpt-6-luna` 분기가 그대로 보이니, 물으면 "연결 환경에 맞춘 설정"이라고만 답합니다.
- 값 바꾸기판(2026-10-18): 객관식·성공 기준 없이 완성 코드를 주고, CHANGE_HERE 5곳(2.1 답 길이, 3.1 앞 대화 없이 "그곳" 묻기, 3.2 첫 질문의 나라, 4.1 시스템 메시지, 4.2 조각 구분 표시)에서 값을 바꿔 봅니다. "무슨 일이 생기나"는 사외 OpenAI(gpt-6-luna)로 기준판·바꾼 판을 두 번씩 실행한 결과입니다.
- temperature(2.1 요청 본문의 0.5)는 gpt-6-luna에서 요청에서 빠져 바꿔도 아무 일이 없으므로 CHANGE_HERE로 쓰지 않았습니다. 사내 경로에서는 그대로 보내집니다.
- CodeMate가 노트북에서 선택한 셀을 읽는지는 강의장에서 아직 확인하지 않았습니다. 못 읽으면 셀 코드와 오류 메시지를 채팅에 붙여 넣게 합니다.

</details>

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 요청을 직접 보내기

`requests`라는 도구로 LLM 서버 주소에 요청을 직접 보냅니다.
요청 본문(payload)은 JSON(이름과 값을 짝지어 적는 데이터 형식)으로 만들고, 응답도 JSON으로 받습니다.

코드는 완성되어 있습니다. 준비 코드부터 차례로 실행하고, 출력된 응답 JSON을 읽어 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 아래 줄의 질문에서 답의 길이를 바꿔 다시 보내고, 응답의 토큰 수를 비교합니다. 요청 본문 셀 아래쪽의 `if` 부분은 연결 환경에 맞춰 설정을 조금 바꾸는 코드라 건너뛰어도 됩니다.

**왜 배우나:** 3장의 SDK와 4장의 LangChain도 속에서는 이런 요청을 보냅니다. 날것의 요청을 한 번 보아 두면, API 오류가 났을 때 어디부터 살펴야 할지 알 수 있습니다.

준비 코드입니다. 필요한 도구를 불러오고, 요청을 보낼 주소·모델 이름·인증 정보를 준비합니다. 접혀 있으니 펼치지 않아도 됩니다. 실행만 합니다.

In [ ]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv
import config  # noqa: F401  .env의 COURSE_ENV(사내/사외)에 맞춰 연결 방식을 채운다

load_dotenv()

provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
if provider == "openai":
    api_base_url = "https://api.openai.com/v1/chat/completions"
    model = os.getenv("OPENAI_MODEL")
    api_key = os.getenv("OPENAI_API_KEY")
    if not model or not api_key:
        raise RuntimeError("OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
elif provider == "samsung":
    # 사내 게이트웨이 주소·모델·인증 헤더는 config/samsung.py에 모아 두었다.
    from config.samsung import llm_config

    samsung = llm_config()
    api_base_url = f"{samsung['base_url'].rstrip('/')}/chat/completions"
    model = samsung["model"]
    headers = {**samsung["headers"], "Content-Type": "application/json"}
else:
    raise ValueError(f"지원하지 않는 LLM_PROVIDER: {provider}")

<details><summary>강사용 메모</summary>

원본: `src/raw/Chapter01/01-llm-api/01_requests.py`. 보이는 셀은 원본 30~54줄, 참고 코드는 57~83줄입니다.

</details>

### 2.1 요청 본문 만들기

**✍️ CHANGE_HERE: "3문장으로"를 "한 문장으로"로 바꾸면 응답의 토큰 수(usage)는 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `3문장으로` 자리에 복사해 붙여 넣기

```text
한 문장으로
```

바꾸기 전에 2.2 출력의 `usage` 숫자(`prompt_tokens`, `completion_tokens`)를 적어 둡니다. 바꾼 뒤 이 셀과 2.2의 코드 셀을 차례로 다시 실행해 비교하고, 확인했으면 되돌려 두 셀을 다시 실행합니다.

In [ ]:
payload = {
    "model": model,
    "messages": [
        # ✍️ CHANGE_HERE: "3문장으로"를 "한 문장으로"로 바꾸면 응답의 토큰 수(usage)는 어떻게 달라질까요?
        {"role": "user", "content": "LLM이 무엇인지 3문장으로 설명해줘."},
    ],
    "temperature": 0.5,
    "stream": False,
}
if provider == "samsung":
    payload["include_reasoning"] = False
elif model == "gpt-6-luna":
    # GPT-6 Luna의 Chat Completions는 기본 추론 설정에서 temperature를 받지 않는다.
    payload.pop("temperature")
    payload["reasoning_effort"] = "none"

<details><summary>무슨 일이 생기나</summary>

두 번씩 실행한 예입니다. 숫자는 실행할 때마다 조금씩 다릅니다.

| 질문 | 답 | `prompt_tokens` | `completion_tokens` |
|---|---|---|---|
| 3문장으로 설명해줘 | 세 문장 | 20 | 92, 94 |
| 한 문장으로 설명해줘 | 한 문장 | 19 | 31, 36 |

- 답이 한 문장으로 짧아지자 출력 토큰(`completion_tokens`)이 3분의 1 정도로 줄었습니다. 입력 토큰(`prompt_tokens`)은 질문 글자가 조금 바뀐 만큼만 달라졌습니다.
- 비용과 응답 시간은 입력·출력 토큰 수로 정해집니다. 답의 길이나 형식을 정해 주면 출력 토큰이 줄어 비용과 시간도 줄어듭니다.

</details>

### 2.2 보내고 응답 받기

In [ ]:
new_response = requests.post(api_base_url, headers=headers, json=payload, timeout=30)
new_response.raise_for_status()
recv_json = new_response.json()

print("### 응답 Json")
pprint(recv_json)
print("### 결과 메시지")
print(recv_json["choices"][0]["message"]["content"])

### 2.3 참고: 다른 요청 방식(Responses API)

실행되지 않는 참고 코드입니다. 읽어만 봅니다.

```python
# OpenAI Responses API 엔드포인트
# - 기존 Chat Completions(/v1/chat/completions)의 후속 API
# - input 필드 하나로 단순 문자열과 대화 내역 모두 처리 가능
api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

if not api_key:
    raise RuntimeError('OPENAI_API_KEY 환경변수를 설정해주세요.')

url = 'https://api.openai.com/v1/responses'
headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}

payload = {
    'model': model,
    'input': 'LLM이 무엇인지 3문장으로 설명해줘.',
}

response = requests.post(url, headers=headers, json=payload, timeout=30)
response.raise_for_status()

data = response.json()
pprint(data)
```

**볼 지점**
- "### 결과 메시지" 아래 줄이 모델의 답변 텍스트입니다.
- "### 응답 Json" 아래에서 `choices` → `message` → `content`를 따라가면 같은 텍스트가 있습니다. 코드는 이 경로로 답을 꺼냈습니다.
- `usage`를 찾아봅니다. `prompt_tokens`는 입력 토큰 수, `completion_tokens`는 출력 토큰 수, `total_tokens`는 둘의 합입니다.
- 숫자는 실행할 때마다 다릅니다. `usage`가 보이지 않으면 "이 환경에서는 제공되지 않음"이라고 적어 둡니다.

<details><summary>강사용 메모</summary>

- 원본 9~28줄의 분기에서 `.env`의 `LLM_PROVIDER` 값에 따라 주소·모델·헤더가 정해집니다. 기본값은 samsung입니다.
- 원본 38~43줄(요청 본문 셀 아래쪽 if): samsung이면 `include_reasoning=False`를 더하고, openai이면서 모델이 `gpt-6-luna`이면 temperature를 빼고 `reasoning_effort='none'`을 넣습니다.
- 사내 게이트웨이 응답에 `usage`가 들어오는지는 아직 확인하지 않았습니다. 토큰 수는 환경마다 다릅니다.
- 2.1 CHANGE_HERE의 결과 글은 응답에 `usage`가 있다고 보고 썼습니다(사외 gpt-6-luna, 출력 토큰 92·94 → 31·36). 사내에서 `usage`가 없으면 답의 길이(세 문장 → 한 문장)만 비교하게 합니다. 사내 모델이 생각 토큰을 `completion_tokens`에 섞어 세면 차이가 덜 뚜렷할 수 있습니다(미확인).

</details>

## 3. SDK로 한 번 묻기, 이어 묻기, 조각으로 받기

이번에는 OpenAI SDK(개발 도구 묶음)로 같은 일을 합니다. 2장에서 손으로 만든 주소·헤더·JSON 처리를 `client`(요청을 대신 보내 주는 객체)가 맡습니다. 요청 방식은 `messages` 대신 `input`에 내용을 담는 Responses API(새 요청 방식)입니다.

한 번 묻기(Single Turn), 이어서 묻기(Multi Turn), 조각으로 받기(Streaming)를 차례로 실행합니다. 한 번 묻기와 이어서 묻기에서는 `# ✍️ CHANGE_HERE:` 아래 줄의 질문을 바꿔, 앞 대화가 있을 때와 없을 때 답이 어떻게 다른지 봅니다. 요청을 네 번 보내므로 답이 모두 나오기까지 조금 걸립니다.

**왜 배우나:** 모델은 앞 대화를 기억하지 않습니다. "이어서 묻기" 셀에서 코드가 이전 대화를 매번 다시 넣어 보내는 모습을 직접 봅니다. 메모리나 Checkpointer(대화 상태를 저장해 두는 장치) 같은 기능은 이 "다시 넣기"를 도구가 대신 해 주는 방법입니다.

이 장의 준비 코드입니다. 요청을 대신 보내 줄 `client`와 모델 이름을 만듭니다. 내용은 몰라도 되니 실행만 하고 넘어갑니다.

In [ ]:
import os
from collections import Counter
from pprint import pprint

from dotenv import load_dotenv
import config  # noqa: F401  .env의 COURSE_ENV(사내/사외)에 맞춰 연결 방식을 채운다
from openai import OpenAI

load_dotenv()

provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
if provider == "openai":
    model = os.getenv("OPENAI_MODEL")
    api_key = os.getenv("OPENAI_API_KEY")
    if not model or not api_key:
        raise RuntimeError("OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.")
    client = OpenAI(api_key=api_key)
elif provider == "samsung":
    # 사내 게이트웨이 주소·모델·인증 헤더는 config/samsung.py에 모아 두었다.
    from config.samsung import llm_config

    samsung = llm_config()
    model = samsung["model"]
    client = OpenAI(
        api_key="samsung-gateway",  # 실제 사내 인증은 x-dep-ticket 헤더에서 처리
        base_url=samsung["base_url"],
        default_headers=samsung["headers"],
    )
else:
    raise ValueError(f"지원하지 않는 LLM_PROVIDER: {provider}")


# OpenAI SDK 예제는 사외에서는 공식 API, 사내에서는 호환 게이트웨이를 사용한다.

<details><summary>강사용 메모</summary>

원본: `src/raw/Chapter01/01-llm-api/02_openai.py`. 한 번 묻기 34~42줄, 이어서 묻기 44~58줄, 조각으로 받기 61~74줄, 참고 77~89줄.
- 노트북에서만 손본 것(patch): 참고 코드 위 `top_p` 주석. 원본의 "확률 상위 p% 토큰만 사용. temperature와 동시 조정은 권장하지 않음"이 강의의 누적 확률 설명, 두 값을 함께 넣은 아래 코드와 어긋나서 고쳤습니다(2026-09-29).

</details>

### 3.1 한 번 묻기

**✍️ CHANGE_HERE: 질문을 "그곳의 인구는 어느정도야?"로 바꾸면 모델은 어떻게 답할까요?**

**바꿔 볼 값** — 바로 아래 줄의 `LLM이 무엇인지 3문장으로 설명해줘.` 자리에 복사해 붙여 넣기

```text
그곳의 인구는 어느정도야?
```

이 셀만 다시 실행합니다. 확인했으면 원래 질문으로 되돌립니다.

In [ ]:
# ── Single Turn ───────────────────────────────────────────────────────────────
# input에 문자열을 직접 전달하면 사용자 메시지로 처리
response = client.responses.create(
    model=model,
    # ✍️ CHANGE_HERE: 질문을 "그곳의 인구는 어느정도야?"로 바꾸면 모델은 어떻게 답할까요?
    input="LLM이 무엇인지 3문장으로 설명해줘.",
)
print(response.output_text)
print()

<details><summary>무슨 일이 생기나</summary>

두 번 실행해 두 번 모두 되물었습니다.

```
어느 곳을 말씀하시는 건가요? 지역 이름을 알려주시면 인구를 확인해 드릴게요.
```

- 이 요청에는 질문 한 줄만 들어 있어서, 모델은 "그곳"이 어디인지 알 수 없습니다. 모델은 이번 요청에 넣은 내용만 봅니다.
- 3.2에서는 같은 질문을 앞 대화(대한민국의 수도는? → 서울)와 함께 보냅니다.

</details>

### 3.2 이어서 묻기

**✍️ CHANGE_HERE: "대한민국"을 "일본"으로 바꾸면 두 번째 질문의 답은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `대한민국` 자리에 복사해 붙여 넣기

```text
일본
```

이 셀만 다시 실행합니다. 두 번째 질문 줄은 그대로 둡니다. 확인했으면 되돌립니다.

In [ ]:
# ── Multi Turn ────────────────────────────────────────────────────────────────
# input에 역할(role)이 있는 메시지 리스트를 전달하면 대화 맥락 유지
# developer: 시스템 지시 / user: 사용자 입력 / assistant: AI 응답
context = [
    {
        "role": "developer",
        "content": "너는 단답형 답변 챗봇이야. 경어체를 유지하면서, 가능하면 단답형으로 답변해줘.",
    },
    # ✍️ CHANGE_HERE: "대한민국"을 "일본"으로 바꾸면 두 번째 질문의 답은 어떻게 될까요?
    {"role": "user", "content": "대한민국의 수도는 어디지?"},
]
response_1 = client.responses.create(model=model, input=context)
print(response_1.output_text)

# 이전 응답(response_1.output)을 context에 추가해 대화를 이어감
context += response_1.output
context += [{"role": "user", "content": "그곳의 인구는 어느정도야?"}]
response_2 = client.responses.create(model=model, input=context)
print(response_2.output_text)

<details><summary>무슨 일이 생기나</summary>

두 번씩 실행한 예입니다.

| 첫 질문 | 첫 답 | 두 번째 답(질문은 그대로) |
|---|---|---|
| 대한민국의 수도는 어디지? | 서울입니다. | 서울의 인구는 약 930만 명입니다. |
| 일본의 수도는 어디지? | 도쿄입니다. | 도쿄도 기준 약 1,400만 명입니다. |

- 두 번째 질문은 한 글자도 바꾸지 않았는데, 답이 서울 인구에서 도쿄 인구로 바뀌었습니다. 코드가 첫 질문과 첫 답을 `context`에 붙여 다시 보냈고, 모델은 그 앞 대화를 읽고 "그곳"을 정합니다.
- 모델이 앞 대화를 기억하는 것이 아닙니다. 앞 대화를 다시 넣어 보내는 코드(`context += ...`)가 맥락을 만듭니다. 3.1에서 앞 대화 없이 물었을 때는 되물었습니다.
- 인구 숫자와 덧붙이는 설명(수도권 인구 등)은 실행할 때마다 조금씩 다릅니다.

</details>

### 3.3 조각으로 받기

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()을 사용하면 LLM이 생성하는 토큰을 실시간으로 받을 수 있음
# output_text.delta 이벤트에 실제 텍스트 조각이 담겨 있음
with client.responses.stream(
    model=model,
    input="LLM이 무엇인지 3문장으로 설명해줘.",
) as stream:
    event_types = []
    for event in stream:
        event_types.append(event.type)
        if "output_text.delta" in event.type:
            print(event.delta, end="")
    print()
    pprint(Counter(event_types))

### 3.4 참고: temperature와 top_p

실행되지 않는 참고 코드입니다. 읽어만 봅니다.

```python
# temperature: 0에 가까울수록 일관된 답변, 높을수록 창의적(최대 2.0)
# top_p: 확률이 높은 후보부터 더한 누적 확률이 top_p에 닿을 때까지의 후보 중에서만 고름
# 보통 temperature와 top_p 중 하나만 바꾼다. 아래는 두 값을 함께 넣어 보인 예시
# 주의: gpt-5 계열은 temperature 미지원
response = client.responses.create(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
    temperature=0.2,
    top_p=0.1,
)
print(response.output_text)
```

**볼 지점**
- 첫 답은 "LLM이 무엇인지 3문장으로 설명해줘."에 대한 답입니다. 2장에서 이 질문으로 처음 받은 답과 문장이 똑같은지 비교해 봅니다.
- "대한민국의 수도는 어디지?"에 짧은 존댓말로 답했는지 봅니다. 이 규칙은 `developer` 메시지에 들어 있습니다.
- "그곳의 인구는 어느정도야?"에 서울의 인구로 답했는지 봅니다. 앞 대화를 `context`에 붙여 다시 보냈기 때문입니다.
- 마지막 답은 조각으로 이어서 출력됩니다. 맨 아래 `Counter(...)`에서 `response.output_text.delta` 옆의 수가 텍스트 조각 이벤트가 온 횟수입니다.

<details><summary>강사용 메모</summary>

- 사내 게이트웨이가 Responses API(`client.responses`)를 지원하는지는 아직 확인하지 않았습니다. 사내에서 오류가 나면 CodeMate 설명만 하고 넘어갑니다.
- 사내 경로에서는 인증을 헤더로 하므로 `api_key` 자리에 이름만 채웁니다(원본 23줄, 접힌 준비 코드 안).
- 3.1·3.2 CHANGE_HERE의 결과 글은 사외 gpt-6-luna로 두 번씩 실행한 결과입니다(3.1은 두 번 모두 "어느 곳을 말씀하시는 건가요?"처럼 되물음, 3.2는 두 번 모두 도쿄 인구로 답함). 사내 모델은 3.1에서 되묻지 않고 짐작해 답할 수도 있습니다(미확인). 그래도 "그곳"이 어디인지 정할 근거가 요청 안에 없다는 점은 같습니다.

</details>

## 4. LangChain으로 호출하기

같은 질문을 LangChain(LLM 호출을 묶어 쓰게 해 주는 도구)으로 보냅니다. 모델 준비는 접힌 준비 코드 안의 `llm = get_llm()` 한 줄로 끝납니다. 2·3장에서 길게 쓰던 준비 코드가 이 함수 하나에 모여 있습니다.

완성된 코드를 실행해 봅니다. 3장에서 한 일과 짝이 맞습니다. 역할을 나눈 메시지는 `SystemMessage`·`HumanMessage`, 한 번에 받기는 `invoke()`, 조각으로 받기는 `stream()`입니다. LangChain은 다음 시간에 자세히 배웁니다.

**왜 배우나:** 앞으로 만드는 Agent도 모두 이 `get_llm()`과 메시지 목록으로 모델을 호출합니다.

준비 코드는 메시지 종류(`SystemMessage`, `HumanMessage`)를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 펼쳐서 `get_llm()` 줄만 확인해 봐도 좋습니다. 실행을 잊지 않습니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()


llm = get_llm()

<details><summary>강사용 메모</summary>

원본: `src/solution/Chapter01/01-llm-api/03_langchain.py`(가독성 우선으로 솔루션을 씀). 받은 원본 `src/raw/…/03_langchain.py`는 TODO 3곳에 정답이 채워져 있고, 스트리밍이 `print(chunk)`, 8~11줄에 정리되지 않은 메모가 있습니다. `get_llm()`은 `src/config/llm.py`에 있습니다.
- 2026-09-28 결정: LangChain은 1-3에서 배우므로 1-2에서는 과제가 아니라 예제로 실행만 합니다(커리큘럼 1.2 "get_llm() 래퍼를 통한 모델 호출 일원화").

</details>

### 4.1 역할 메시지로 한 번에 받기

**✍️ CHANGE_HERE: 역할을 "너는 항상 번호 목록(1. 2. 3.)으로만 답하는 AI 튜터야."로 바꾸면 답은 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `너는 초보자를 위한 AI 튜터야.` 자리에 복사해 붙여 넣기

```text
너는 항상 번호 목록(1. 2. 3.)으로만 답하는 AI 튜터야.
```

이 셀을 다시 실행합니다. 4.2도 이 `messages`를 쓰므로, 확인했으면 되돌리고 이 셀을 다시 실행한 뒤 4.2로 넘어갑니다.

In [ ]:
# ── 기본 invoke ───────────────────────────────────────────────────────────────
# SystemMessage: LLM의 역할/행동 지침 설정
# HumanMessage: 사용자 입력
messages = [
    # ✍️ CHANGE_HERE: 역할을 "너는 항상 번호 목록(1. 2. 3.)으로만 답하는 AI 튜터야."로 바꾸면 답은 어떻게 달라질까요?
    SystemMessage(content="너는 초보자를 위한 AI 튜터야."),
    HumanMessage(content="LLM이 무엇인지 3문장으로 설명해줘."),
]

response = llm.invoke(messages)
print(response.content)
print()

<details><summary>무슨 일이 생기나</summary>

두 번 실행해 두 번 모두 번호 목록으로 답했습니다.

바꾸기 전(한 문단):

```
LLM은 ‘대규모 언어 모델(Large Language Model)’의 약자로, 많은 글을 학습해 사람의 언어를 이해하고 만들어 내는 AI예요. 질문에 답하거나 글을 요약하고 번역하는 등 다양한 일을 할 수 있어요. 다만 그럴듯하지만 틀린 답을 할 수도 있으니 중요한 정보는 확인해야 해요.
```

바꾼 뒤(번호 목록):

```
1. LLM은 대규모 텍스트 데이터를 학습해 사람의 언어를 이해하고 생성하는 인공지능 모델입니다.
2. 질문에 답하거나 글을 요약하고 번역하는 등 다양한 언어 작업을 수행할 수 있습니다.
3. 다만 학습한 정보가 오래되었거나 불완전할 수 있어, 답변을 항상 사실로 받아들이기보다는 확인하는 것이 좋습니다.
```

- 질문(`HumanMessage`)은 그대로인데 답의 모양이 바뀌었습니다. `SystemMessage`는 답의 역할과 규칙, 출력 형식을 정합니다.
- 셀 출력 위쪽 디버그 정보의 `prompt_tokens`도 36에서 48로 늘었습니다. 시스템 메시지도 요청마다 입력 토큰으로 함께 보내집니다.

</details>

### 4.2 조각으로 받기

**✍️ CHANGE_HERE: end=""를 end=" | "로 바꾸면 출력은 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `end=""` 자리에 복사해 붙여 넣기

```text
end=" | "
```

이 셀을 다시 실행합니다. 확인했으면 되돌립니다.

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()은 토큰 단위로 응답을 yield함
# end='' 로 줄바꿈 없이 실시간 출력
for chunk in llm.stream(messages):
    # ✍️ CHANGE_HERE: end=""를 end=" | "로 바꾸면 출력은 어떻게 달라질까요?
    print(chunk.content, end="", flush=True)
print()

<details><summary>무슨 일이 생기나</summary>

두 번 실행한 예의 앞부분입니다.

```
 | LL | M | 은 |  ‘ | 대 | 규 | 모 |  언 | 어 |  모델 | ’ | 의 |  약 | 자로 | , |  방 | 대한 |  글 | 을 |  학 | 습 | 해 | ...
```

- `print`가 조각 하나를 출력할 때마다 끝에 ` | `를 붙였습니다. `|` 사이가 한 번에 받은 조각 하나입니다.
- 조각은 단어보다 작습니다. `LLM`이 `LL`과 `M`으로, `규모`가 `규`와 `모`로 나뉘어 왔습니다. 모델은 토큰 단위로 답을 만들고, 조각으로 받기는 만든 토큰을 바로바로 보내 줍니다. 토큰은 단어와 딱 맞지 않습니다.
- 글자가 든 조각은 81개, 85개로, 셀 아래쪽 디버그 정보의 `output_tokens`(85, 89)와 거의 같았습니다. 조각 하나가 대략 토큰 하나입니다.
- 맨 앞과 맨 끝에 `|`만 이어진 곳은 글자 없이 온 조각입니다.

</details>

**볼 지점**
- 두 셀 모두 같은 `messages`를 보냅니다. 첫 셀은 답을 한 번에, 둘째 셀은 조각으로 받습니다.
- 둘째 셀의 `print(chunk.content, end="", flush=True)`가 조각의 텍스트만 줄바꿈 없이 바로 보여 줍니다.

<details><summary>강사용 메모</summary>

- 사내 환경에서 조각에 `usage_metadata`가 오는지는 아직 확인하지 않았습니다.
- 4.2 CHANGE_HERE의 결과 글(조각이 글자·글자 묶음 단위, 조각 수 ≈ `output_tokens`)은 사외 gpt-6-luna 결과입니다. 사내 게이트웨이가 여러 토큰을 묶어 보내면 조각이 더 클 수 있습니다(미확인).

</details>

## 5. 정리

같은 질문을 세 가지 방법으로 보냈습니다.

| 장 | 보내는 방법 | 답변 텍스트를 꺼내는 곳 |
|---|---|---|
| 2. 요청을 직접 보내기 | 주소·헤더·JSON을 직접 만들어 보냄 | `choices[0].message.content` |
| 3. SDK로 묻기 | SDK의 `client`가 대신 보냄 | `output_text` |
| 4. LangChain으로 호출하기 | `get_llm()`으로 준비한 모델에 메시지 목록을 보냄 | `content` |

방법은 달라도 "메시지와 설정값을 보내고, 응답에서 답변 텍스트를 꺼낸다"는 흐름은 같습니다. 앞 대화를 이어 가려면 코드가 이전 대화를 다시 넣어 보내야 한다는 점도 확인했습니다.

값을 바꿔 보며 이런 것도 확인했습니다.

- 답의 길이를 정해 주면 출력 토큰이 줄어듭니다. 비용과 시간은 토큰 수로 정해집니다.
- 앞 대화 없이 "그곳"을 물으면 모델은 되묻습니다. 모델은 이번 요청에 넣은 것만 봅니다.
- 시스템 메시지로 답의 형식(번호 목록 등)을 정할 수 있습니다.
- 조각으로 받는 답은 단어보다 작은 토큰 단위로 옵니다.